In [4]:
import os
import json
import sys
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR

while not (PROJECT_ROOT / "scripts" / "train.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MEMBER = "BestVerie"
ENV_ID = "ALE/Boxing-v5"
SEED = 42
TOTAL_TIMESTEPS = 100_000
EVAL_EPISODES = 10

TRAIN_SCRIPT = PROJECT_ROOT / "scripts" / "train.py"
BASE_DIR = PROJECT_ROOT / "results" / MEMBER
MODEL_DIR = BASE_DIR / "models"
LOG_DIR = BASE_DIR / "logs"
TABLE_DIR = BASE_DIR / "tables"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook dir :", NOTEBOOK_DIR)
print("Project root :", PROJECT_ROOT)
print("Train script :", TRAIN_SCRIPT)
print("Base dir     :", BASE_DIR)
print("Train exists :", TRAIN_SCRIPT.exists())

Notebook dir : /Users/mwizera/Downloads/boxing_dqn_agent/experiments
Project root : /Users/mwizera/Downloads/boxing_dqn_agent
Train script : /Users/mwizera/Downloads/boxing_dqn_agent/scripts/train.py
Base dir     : /Users/mwizera/Downloads/boxing_dqn_agent/results/BestVerie
Train exists : True


In [8]:
experiments = [

    # 1. Baseline CNN
    {
        "name": "boxing_exp01_baseline_cnn",
        "policy": "CnnPolicy",
        "learning_rate": 1e-4,
        "gamma": 0.99,
        "batch_size": 32,
        "exploration_fraction": 0.10,
        "exploration_initial_eps": 1.0,
        "exploration_final_eps": 0.05,
        "gradient_steps": 1
    },

    # 2. Small batch 
    {
        "name": "boxing_exp02_small_batch_cnn",
        "policy": "CnnPolicy",
        "learning_rate": 1e-4,
        "gamma": 0.99,
        "batch_size": 16,
        "exploration_fraction": 0.10,
        "exploration_initial_eps": 1.0,
        "exploration_final_eps": 0.05,
        "gradient_steps": 1
    },

   
]

print(f"Prepared {len(experiments)} Boxing experiments")

Prepared 2 Boxing experiments


In [6]:
# Helper to run one experiment through scripts/train.py
import sys
import subprocess
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR

while not (PROJECT_ROOT / "scripts" / "train.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_SCRIPT = PROJECT_ROOT / "scripts" / "train.py"

print("Notebook dir :", NOTEBOOK_DIR)
print("Project root :", PROJECT_ROOT)
print("Train script :", TRAIN_SCRIPT)
print("Exists?      :", TRAIN_SCRIPT.exists())

def run_experiment(exp: dict):
    cmd = [
        sys.executable, str(TRAIN_SCRIPT),
        "--member", MEMBER,
        "--experiment", exp["name"],
        "--env-id", ENV_ID,
        "--policy", exp.get("policy", "CnnPolicy"),
        "--total-timesteps", str(TOTAL_TIMESTEPS),
        "--seed", str(SEED),
        "--learning-rate", str(exp["learning_rate"]),
        "--gamma", str(exp["gamma"]),
        "--batch-size", str(exp["batch_size"]),
        "--buffer-size", str(exp.get("buffer_size", 50_000)),
        "--learning-starts", str(exp.get("learning_starts", 5_000)),
        "--train-freq", str(exp.get("train_freq", 4)),
        "--gradient-steps", str(exp.get("gradient_steps", 1)),
        "--target-update-interval", str(exp.get("target_update_interval", 10_000)),
        "--exploration-initial-eps", str(exp.get("exploration_initial_eps", 1.0)),
        "--exploration-final-eps", str(exp.get("exploration_final_eps", 0.05)),
        "--exploration-fraction", str(exp.get("exploration_fraction", 0.10)),
        "--eval-freq", "10000",
        "--eval-episodes", str(EVAL_EPISODES),
    ]

    print("\nRunning:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True
    )

    print("\nSTDOUT:\n", result.stdout)
    if result.stderr:
        print("\nSTDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Experiment failed with exit code {result.returncode}")

Notebook dir : /Users/mwizera/Downloads/boxing_dqn_agent/experiments
Project root : /Users/mwizera/Downloads/boxing_dqn_agent
Train script : /Users/mwizera/Downloads/boxing_dqn_agent/scripts/train.py
Exists?      : True


In [7]:
#  Run all 10 experiments
for exp in experiments:
    run_experiment(exp)

print('All experiments finished.')



Running:
/opt/homebrew/anaconda3/bin/python /Users/mwizera/Downloads/boxing_dqn_agent/scripts/train.py --member BestVerie --experiment best_exp01_baseline --env-id ALE/Boxing-v5 --policy CnnPolicy --total-timesteps 100000 --seed 42 --learning-rate 0.0001 --gamma 0.99 --batch-size 32 --buffer-size 50000 --learning-starts 5000 --train-freq 4 --gradient-steps 1 --target-update-interval 10000 --exploration-initial-eps 1.0 --exploration-final-eps 0.05 --exploration-fraction 0.1 --eval-freq 10000 --eval-episodes 10

STDOUT:
 

STDERR:
 Traceback (most recent call last):
  File "/Users/mwizera/Downloads/boxing_dqn_agent/scripts/train.py", line 21, in <module>
    import ale_py  # noqa: F401
    ^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'ale_py'



RuntimeError: Experiment failed with exit code 1

In [ ]:
#  Read evaluation summaries into one table
rows = []
missing = []

for exp in experiments:
    eval_path = BASE_DIR / exp['name'] / f"{exp['name']}_eval.json"
    if eval_path.is_file():
        with open(eval_path, 'r', encoding='utf-8') as f:
            rows.append(json.load(f))
    else:
        missing.append(str(eval_path))

if missing:
    print('Missing evaluation files:')
    for m in missing:
        print(' -', m)

results_df = pd.DataFrame(rows).sort_values('mean_reward', ascending=False).reset_index(drop=True)
results_df


In [ ]:
#  Save results table
results_csv = TABLE_DIR / f'{MEMBER}_boxing_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'Saved results table to: {results_csv}')
results_df


In [ ]:
# Identify Best's best model
if results_df.empty:
    raise ValueError('No results found. Run the experiments first.')

best_row = results_df.iloc[0]

best_experiment_name = best_row['experiment']
best_mean_reward = best_row['mean_reward']
best_model_path = best_row['model_path']

print('Best experiment:', best_experiment_name)
print('Best mean reward:', best_mean_reward)
print('Best model path:', best_model_path)

In [ ]:
#  Save best summary JSON
best_summary_path = TABLE_DIR / f'{MEMBER}_best_summary.json'
with open(best_summary_path, 'w', encoding='utf-8') as f:
    json.dump({'member': MEMBER, 'best_experiment': best_experiment_name, 'best_mean_reward': float(best_mean_reward), 'best_model_path': best_model_path}, f, indent=2)
print(f'Saved best summary to: {best_summary_path}')


In [ ]:
#  Visualization 1 — mean reward by experiment
_df = results_df.copy().sort_values('mean_reward', ascending=False)
plt.figure(figsize=(10, 4))
plt.bar(_df['experiment'], _df['mean_reward'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Mean reward (evaluation)')
plt.title('Best — DQN Boxing mean reward by experiment')
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2 — training time vs performance
plt.figure(figsize=(6, 4))
plt.scatter(results_df['train_minutes'], results_df['mean_reward'])
plt.xlabel('Train minutes')
plt.ylabel('Mean reward')
plt.title('Training time vs performance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import json
from pathlib import Path
import pandas as pd

results = []

base = PROJECT_ROOT / "results" / "best"

for p in base.glob("best_exp*/best_exp*_eval.json"):
    with open(p) as f:
        data = json.load(f)
        results.append(data)

df = pd.DataFrame(results)
df = df.sort_values("mean_reward", ascending=False)

df